# 📝 과제 01: 의약품 상담 도우미의 답변 품질 평가

## 만들 결과

약국 체인 본사의 **의약품 상담 도우미**가 답변 프롬프트를 바꾸려고 합니다. 같은 골든셋(정답셋) 질문 6개에 현재 프롬프트 `v1`과 개선안 프롬프트 `v2`로 RAG를 실행한 기록이 저장되어 있습니다. 두 버전의 답변을 생성 품질 지표 세 개로 평가하고, 개선안을 출시할지 정합니다.

- 골든셋과 실행 기록을 합쳐 평가 샘플 12개(질문 6개 x 버전 2개)를 만듭니다.
- Faithfulness(답변 충실도)·Answer Relevancy(답변 관련성)·Answer Correctness(답변 정확도)를 동시에 계산해 점수 행을 쌓고, 지표별 표를 만듭니다.
- 교안 01 6-3의 두 지표 함께 읽기 표를 함수로 옮겨 답변마다 결함 라벨을 붙입니다.
- 버전별 평균과 문제 답변 수를 비교하고, 출시 판정 규칙을 함수로 옮겨 판정합니다.
- 판정을 결정 기록으로 남깁니다.

교안 01의 세 지표 계산과 6-3의 두 지표 함께 읽기를 의약품 상담 답변에 적용합니다. 라벨 이름은 교안 01 본실습의 변형 이름(`근거대로`, `근거에_없는_내용_추가`, `숫자_조건_바꿈`, `질문과_다른_이야기`)과 같습니다. 교안은 결함 종류를 알고 지표를 확인했고, 과제는 결함 종류를 모르는 답변에서 지표로 결함을 찾습니다.

의학적 조언이 아니라 수업용 평가 자료이며, 실제 복용은 의사·약사와 상의해야 합니다.

**두 버전의 차이**

상담 도우미 팀은 "아이 용량을 물었는데 약 설명만 나왔다", "알려 준 용량이 안내문과 달랐다"는 고객 의견을 받고 `v2`를 준비했습니다. 두 프롬프트에서 달라진 지시는 두 가지입니다.

| 바뀐 지시 | `v1`(현재) | `v2`(개선안) |
|---|---|---|
| 답하는 방식 | 검색한 안내문을 요약해 답한다 | 질문이 묻는 부분(용량·횟수·시점·병용·사용 대상 등)을 안내문에서 찾아 답하고, 숫자는 안내문에 적힌 그대로 쓴다 |
| 덧붙이는 내용 | 요약한 내용만 답한다 | 복용할 때 알아 두면 좋은 팁이 있으면 한두 문장 덧붙인다 |

- 검색기는 두 버전이 같습니다. 그래서 같은 질문의 두 기록은 검색 문서가 같고, 답변만 다릅니다.
- 버전을 비교할 때는 현재 버전 `v1`을 **기준선**(baseline)으로 삼아, 개선안 `v2`가 기준선보다 나아졌는지 봅니다.
- 평가로 확인할 것은 두 지시의 효과입니다. 첫째 지시로 질문과 다른 답과 숫자가 바뀐 답이 줄었는지, 둘째 지시로 검색 문서에 없는 내용이 끼어들지 않았는지를 봅니다.

## 준비: 데이터

- **자료:** `data/과제01/`의 골든셋 `golden_set.json`(질문 6개와 질문마다 정답 답변)과 미리 저장한 RAG 실행 기록 `rag_records.json`(12건, 답변과 검색 문서). 검색 문서는 식약처 「e약은요」 안내문입니다. 출처와 작성 규칙은 [자료 설명](data/과제01/README.md)에 있습니다.
- **순서:** 평가 샘플 -> 세 지표 계산 -> 지표별 표 -> 판독 라벨 -> 버전 비교 -> 출시 판정 -> 결정 기록
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행하세요.
- **점수:** 평가용 LLM의 판정이라 실행마다 조금씩 달라집니다.
- **다시 실행할 때:** 2절이 네트워크 오류로 멈추면 그 셀만 다시 실행합니다. 결과 목록을 셀 안에서 새로 만들므로 점수가 중복으로 쌓이지 않습니다.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 저장소 최상위 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

os.environ["RAGAS_DO_NOT_TRACK"] = "true"  # ragas가 익명 사용 통계를 외부 서버로 보내지 않게 합니다

# 평가용 LLM: ragas 0.4 지표는 LangChain 모델 대신 OpenAI SDK의 비동기 클라이언트를 받습니다
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-6-luna")  # gpt-5.6-luna처럼 버전에 소수점이 든 이름은 ragas 0.4.3에서 요청 오류가 납니다
eval_client = AsyncOpenAI()

KOREAN_RULE = "모든 출력 문장(질문·주장·이유)은 한국어로 작성하세요."

evaluator_llm = llm_factory(
    eval_model_name,
    client=eval_client,
    max_tokens=8192,            # 응답 길이 상한. 추론 모델은 생각하는 데 쓴 토큰도 여기에 들어가므로 기본값(1024)보다 크게 둡니다
    system_prompt=KOREAN_RULE,  # 이 평가용 LLM을 쓰는 모든 지표에 한국어 규칙이 적용됩니다
)

# Answer Relevancy·Answer Correctness가 문장 사이 의미 유사도를 계산할 때 쓰는 임베딩 모델
evaluator_embeddings = embedding_factory(
    "openai",
    model="text-embedding-3-large",  # ragas 0.4.3 임베딩은 차원 인자를 받지 않아 기본 3072차원을 씁니다
    client=eval_client,
)

print("평가용 LLM:", eval_model_name)

In [ ]:
# 골든셋과 미리 저장한 실행 기록 읽기
golden_set = json.loads((data_dir / "과제01" / "golden_set.json").read_text(encoding="utf-8"))
rag_records = json.loads((data_dir / "과제01" / "rag_records.json").read_text(encoding="utf-8"))
print("골든셋 질문:", len(golden_set), "| 실행 기록:", len(rag_records))

In [ ]:
# 골든셋 질문 6개 (정답 답변은 사람이 안내문 원문을 확인하고 썼습니다)
golden_view = []
for item in golden_set:
    golden_view.append({
        "질문 ID": item["question_id"],
        "제품": item["drug"],
        "질문": item["user_input"],
    })
display(pd.DataFrame(golden_view))

In [ ]:
# 실행 기록 하나의 모양 (검색 문서는 앞부분만)
first = rag_records[0]
print("질문 ID:", first["question_id"], "| 버전:", first["version"], "| 검색 문서 ID:", first["context_doc_ids"])
print("답변:", first["response"])
print("검색 문서 1 앞부분:", first["retrieved_contexts"][0][:120])

## 1. 골든셋과 실행 기록을 평가 샘플로 합치세요

세 지표는 평가 샘플의 네 칸(질문·검색 문서·답변·정답 답변) 가운데 일부를 받는데, 질문과 정답 답변은 골든셋에, 답변과 검색 문서는 실행 기록에 나뉘어 있으므로 질문 ID로 합칩니다.

**작성:** 평가 샘플 목록 `samples`

**목적:** 교안 01 3-3의 평가 샘플(질문·검색 문서·답변·정답 답변)을 두 파일에서 만듭니다. 질문 ID와 버전을 함께 남겨 뒤에서 같은 질문의 두 버전을 나란히 놓습니다.

**구현할 처리**
1. 질문 ID로 골든셋 항목을 찾는 dict `golden_by_id`를 만듭니다.
2. 실행 기록을 파일 순서대로 하나씩 읽어 아래 여섯 키의 dict를 이 순서로 만들어 `samples`에 더합니다.

| 키 | 값 |
|---|---|
| `질문 ID`, `버전` | 실행 기록의 `question_id`, `version` |
| `user_input` | 골든셋의 질문 |
| `retrieved_contexts` | 실행 기록의 검색 문서 본문 목록 |
| `response` | 실행 기록의 답변 |
| `reference` | 골든셋의 정답 답변 |

**반환값:** `samples`(dict 12개 리스트, 실행 기록과 같은 순서)

In [ ]:
# [작성] 골든셋과 실행 기록을 질문 ID로 합쳐 평가 샘플 12개를 만드세요.

# 1. 질문 ID로 골든셋 항목을 찾는 dict
golden_by_id = ...

samples = []
for record in rag_records:
    golden = golden_by_id[record["question_id"]]

    # 2. 여섯 키의 평가 샘플
    samples.append({
        "질문 ID": record["question_id"],
        "버전": record["version"],
        "user_input": ...,
        "retrieved_contexts": ...,
        "response": ...,
        "reference": ...,
    })

In [ ]:
# [자가채점] 샘플 수·키 순서와 값이 두 파일과 같은지 확인합니다.
sample_keys = ["질문 ID", "버전", "user_input", "retrieved_contexts", "response", "reference"]
golden_check = {}
for item in golden_set:
    golden_check[item["question_id"]] = item

assert isinstance(samples, list) and len(samples) == len(rag_records), "실행 기록마다 샘플 하나, 12개를 만드세요. `...`를 채웠는지도 확인하세요."
for sample, record in zip(samples, rag_records):
    key = record["question_id"] + " " + record["version"]
    assert list(sample) == sample_keys, f"샘플의 키는 {sample_keys} 순서입니다."
    assert sample["질문 ID"] == record["question_id"] and sample["버전"] == record["version"], "샘플은 실행 기록과 같은 순서로 만드세요."
    golden = golden_check[record["question_id"]]
    assert sample["user_input"] == golden["user_input"], f"{key}: user_input은 골든셋의 질문입니다."
    assert sample["reference"] == golden["reference"], f"{key}: reference는 골든셋의 정답 답변입니다."
    assert sample["response"] == record["response"], f"{key}: response는 실행 기록의 답변입니다."
    assert sample["retrieved_contexts"] == record["retrieved_contexts"], f"{key}: retrieved_contexts는 검색 문서 ID가 아니라 본문 목록입니다."
print("평가 샘플 확인 완료:", len(samples), "개")

In [ ]:
# 샘플 12개를 한 표로 봅니다 (답변은 앞부분만)
sample_view = []
for sample, record in zip(samples, rag_records):
    sample_view.append({
        "질문 ID": sample["질문 ID"],
        "버전": sample["버전"],
        "답변 앞부분": sample["response"][:45],
        "검색 문서 ID": ", ".join(record["context_doc_ids"]),
    })
display(pd.DataFrame(sample_view))

## 2. 세 지표로 답변 12개를 평가하세요

버전을 비교하려면 모든 답변을 같은 세 지표로 평가해야 하므로, 답변마다 세 지표의 `ascore`를 부르고 지표마다 점수를 한 행씩 쌓습니다.

교안 01 4-2처럼 `ascore(...)`를 `await` 없이 작업 목록에 담고, 골격 2번의 `tqdm_asyncio.gather`가 그 작업들을 동시에 실행합니다. 한꺼번에 너무 많은 요청이 나가지 않도록, 아래 셀의 `run_limited`로 감싸 동시에 `MAX_CONCURRENCY`개까지만 실행합니다.

| 이 절에서 쓰는 코드 | 입력과 반환값 |
|---|---|
| `faithfulness`, `answer_relevancy`, `answer_correctness` | 교안 01 4~6절과 같은 설정의 지표 객체. `ascore(...)` 결과의 `.value`가 점수 |
| `run_limited(작업)` | 세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행하고 결과를 돌려줍니다. 동시에 실행되는 작업은 `MAX_CONCURRENCY`개를 넘지 않습니다 |
| 골격 2번 `tqdm_asyncio.gather(*[run_limited(task) for task in 작업 목록], desc=이름)` 세 줄 | 작업 목록을 동시에 실행하고, 넘긴 순서대로 결과 목록을 돌려줍니다. 진행 막대에 끝난 개수를 보여 줍니다 |

**작성:** 작업 목록에 넣을 `ascore(...)` 호출 세 개와 점수 행 dict

**목적:** 교안 01 4~6절의 "답변마다 `ascore`로 평가하고 지표마다 한 행 쌓기"를 변형 대신 버전 열로 적용합니다.

**구현할 처리**
1. 샘플마다 세 지표의 `ascore(...)`를 `await` 없이 호출해 지표별 작업 목록에 더합니다. 넘기는 칸은 교안 01과 같습니다. Faithfulness는 질문·답변·검색 문서, Answer Relevancy는 질문·답변, Answer Correctness는 질문·답변·정답 답변이고, 값은 그 샘플의 `user_input`, `response`, `retrieved_contexts`, `reference`입니다.
2. 골격 2번의 세 줄이 지표마다 작업 목록을 동시에 실행해 결과 목록을 받습니다.
3. 샘플과 세 결과를 같은 순서로 짝지어, 샘플마다 지표 세 행을 `score_rows`에 더합니다. 행은 `질문 ID`, `버전`, `지표`, `점수` 네 키를 이 순서로 쓰고, 점수는 결과의 `.value`입니다.

**반환값:** `score_rows`(dict 36개 리스트: 샘플 순서대로, 샘플마다 Faithfulness, Answer Relevancy, Answer Correctness 순서의 세 행)

In [ ]:
from ragas.metrics.collections import AnswerCorrectness, AnswerRelevancy, Faithfulness

faithfulness = Faithfulness(llm=evaluator_llm)  # 주장 나누기와 검색 문서 대조에 쓸 평가용 LLM

answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,                # 답변에서 질문을 거꾸로 만들 평가용 LLM
    embeddings=evaluator_embeddings,  # 원래 질문과 만든 질문의 유사도를 계산할 임베딩 모델
)

answer_correctness = AnswerCorrectness(
    llm=evaluator_llm,                # 주장을 나누고 TP·FP·FN으로 분류할 평가용 LLM
    embeddings=evaluator_embeddings,  # 답변과 정답 답변의 의미 유사도를 계산할 임베딩 모델
)

In [ ]:
import asyncio

# 동시에 실행할 평가 작업 수의 상한입니다. gather는 넘긴 작업을 한꺼번에 시작하므로, 작업이 많으면 API 요청 한도(분당 요청·토큰 수)를 넘어 429 오류가 날 수 있습니다.
# 32는 ragas evaluate()의 기본 동시 실행 수(RunConfig의 max_workers, 16)보다 넉넉하게 잡은 값입니다. 429 오류가 나면 줄이고, 키 한도가 넉넉하면 늘립니다.
MAX_CONCURRENCY = 32
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)  # 동시에 실행 중인 작업이 MAX_CONCURRENCY개가 되면 나머지는 자리가 날 때까지 기다립니다


async def run_limited(task):
    """세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행하고 결과를 돌려줍니다."""
    async with semaphore:
        return await task

In [ ]:
# [작성] 답변 12개를 세 지표로 평가해 score_rows에 쌓으세요.
from tqdm.asyncio import tqdm_asyncio

# 1. 지표마다 샘플 12개의 ascore를 기다리지 않고 작업 목록에 담기
faithfulness_tasks = []
relevancy_tasks = []
correctness_tasks = []
for sample in samples:
    faithfulness_tasks.append(...)  # 질문·답변·검색 문서
    relevancy_tasks.append(...)     # 질문·답변
    correctness_tasks.append(...)   # 질문·답변·정답 답변

# 2. 지표마다 동시에 실행하고 결과를 samples 순서대로 받기 (동시에 MAX_CONCURRENCY개까지)
faithfulness_results = await tqdm_asyncio.gather(*[run_limited(task) for task in faithfulness_tasks], desc="Faithfulness")
relevancy_results = await tqdm_asyncio.gather(*[run_limited(task) for task in relevancy_tasks], desc="Answer Relevancy")
correctness_results = await tqdm_asyncio.gather(*[run_limited(task) for task in correctness_tasks], desc="Answer Correctness")

# 3. 샘플마다 지표 세 행 쌓기
score_rows = []
for sample, faithfulness_result, relevancy_result, correctness_result in zip(samples, faithfulness_results, relevancy_results, correctness_results):
    for metric_name, result in [
        ("Faithfulness", faithfulness_result),
        ("Answer Relevancy", relevancy_result),
        ("Answer Correctness", correctness_result),
    ]:
        score_rows.append(...)  # 질문 ID·버전·지표·점수 네 키의 dict
print("점수 행:", len(score_rows))

In [ ]:
# [자가채점] 행 수·키·순서와 점수 범위를 확인합니다. 점수 값은 실행마다 달라지므로 보지 않습니다.
import hashlib

metric_order = ["Faithfulness", "Answer Relevancy", "Answer Correctness"]
assert isinstance(score_rows, list) and len(score_rows) == 3 * len(samples), "샘플마다 지표 세 행, 모두 36행이어야 합니다. `...`를 채웠는지 확인하세요."
for index, row in enumerate(score_rows):
    sample = samples[index // 3]
    assert list(row) == ["질문 ID", "버전", "지표", "점수"], "행은 질문 ID, 버전, 지표, 점수 네 키를 이 순서로 씁니다."
    assert row["질문 ID"] == sample["질문 ID"] and row["버전"] == sample["버전"], "행은 samples 순서대로, 샘플마다 세 행씩 쌓으세요."
    assert row["지표"] == metric_order[index % 3], "한 샘플의 세 행은 Faithfulness, Answer Relevancy, Answer Correctness 순서입니다."
    assert isinstance(row["점수"], float), f"점수에는 결과 객체가 아니라 .value(실수)를 넣으세요: {row}"
    if not pd.isna(row["점수"]):
        assert 0 <= row["점수"] <= 1, f"점수는 0~1 사이여야 합니다: {row}"

# 실행마다 값은 달라도 방향은 바뀌지 않는 비교 9쌍: (더 낮은 점수, 더 높은 점수)
# 어느 답변인지 미리 드러나지 않도록 (지표, 질문 ID, 버전)을 해시 지문으로 적었습니다
def key_fingerprint(metric_name, question_id, version):
    return hashlib.sha256(f"{metric_name}|{question_id}|{version}".encode("utf-8")).hexdigest()[:10]


score_by_fingerprint = {}
for row in score_rows:
    score_by_fingerprint[key_fingerprint(row["지표"], row["질문 ID"], row["버전"])] = row["점수"]
direction_checks = [
    ("1d1fa16ebf", "bf16f9cf45"),
    ("56b7552bef", "f61e280ce3"),
    ("dafe8ed3b9", "25f14de132"),
    ("91ede913c9", "4c09e95c26"),
    ("c1a920598c", "e32e63a631"),
    ("c07a2adf0d", "142afc522f"),
    ("a2a3c93d22", "70999c7689"),
    ("c1a920598c", "9edf483f9f"),
    ("c07a2adf0d", "91ede913c9"),
]
for low_key, high_key in direction_checks:
    if pd.isna(score_by_fingerprint[low_key]) or pd.isna(score_by_fingerprint[high_key]):
        print("NaN 점수가 있어 방향 비교 하나를 건너뜁니다. 4절 라벨 표에서 원문 확인 행을 보세요.")
        continue
    assert score_by_fingerprint[low_key] < score_by_fingerprint[high_key], "실행마다 늘 성립하던 점수의 높낮이가 뒤집혔습니다. 세 지표의 ascore에 그 샘플의 칸을 맞게 넘겼는지(Faithfulness: user_input·response·retrieved_contexts, Answer Relevancy: user_input·response, Answer Correctness: user_input·response·reference), 결과 목록과 지표 이름의 짝이 맞는지 확인하세요."
print("점수 행 확인 완료:", len(score_rows), "행")

## 3. 지표별 표와 답변별 점수표를 만드세요

같은 질문의 두 버전을 나란히 놓아야 어느 질문에서 점수가 갈렸는지 보이고, 답변 하나의 세 점수를 한 행에 모아야 4절에서 함께 읽을 수 있으므로 두 가지 모양으로 바꿉니다.

**작성:** 지표별 표 dict `metric_tables`, 답변별 점수표 `answer_table`

**목적:** 교안 01의 `pivot` 표(행은 변형, 열은 질문 ID)를 버전 비교에 맞게 행은 질문 ID, 열은 버전으로 바꿔 만듭니다.

**구현할 처리**
1. `score_rows`를 DataFrame `score_df`로 바꿉니다(골격에 있음).
2. 지표마다 `score_df`에서 그 지표 행만 골라, 행은 `질문 ID`, 열은 `버전`, 값은 `점수`인 표를 `pivot`으로 만들고 소수 둘째 자리로 반올림해 `metric_tables[지표 이름]`에 넣습니다.
3. `answer_table`은 행이 (`질문 ID`, `버전`), 열이 `지표`, 값이 `점수`인 표입니다. 칸마다 점수가 하나라 `pivot`을 쓰고, 행 이름 두 개는 `index=["질문 ID", "버전"]`처럼 리스트로 넘깁니다. 4절에서 기준값과 비교하므로 반올림하지 않습니다.

**반환값:** `metric_tables`(지표 이름 3개를 키, 6행 x 2열 표를 값으로 하는 dict), `answer_table`(12행 x 3열 표)

In [ ]:
# [작성] 지표별 표(행: 질문 ID, 열: 버전)와 답변별 점수표(행: 질문 ID·버전, 열: 지표)를 만드세요.
score_df = pd.DataFrame(score_rows)
metric_names = ["Faithfulness", "Answer Relevancy", "Answer Correctness"]

# 2. 지표마다 그 지표 행만 골라 pivot (소수 둘째 자리)
metric_tables = {}
for metric_name in metric_names:
    metric_tables[metric_name] = ...

# 3. 답변마다 세 지표를 한 행으로 (반올림하지 않음)
answer_table = ...

In [ ]:
# [자가채점] 표 모양을 확인하고, 칸마다 값을 score_rows의 점수와 대조합니다.
question_ids = []
for sample in samples:
    if sample["질문 ID"] not in question_ids:
        question_ids.append(sample["질문 ID"])

assert isinstance(metric_tables, dict) and set(metric_tables) == set(metric_names), "metric_tables의 키는 지표 이름 세 개입니다. `...`를 채웠는지 확인하세요."
for metric_name in metric_names:
    table = metric_tables[metric_name]
    assert isinstance(table, pd.DataFrame), f"{metric_name}: 값은 DataFrame 표입니다."
    assert list(table.index) == question_ids and list(table.columns) == ["v1", "v2"], f"{metric_name}: 행은 질문 ID 6개, 열은 버전 v1·v2인 표입니다."

assert isinstance(answer_table, pd.DataFrame) and answer_table.shape == (len(samples), 3), "answer_table은 답변 12행 x 지표 3열입니다."
assert list(answer_table.index.names) == ["질문 ID", "버전"], "answer_table의 행 이름은 질문 ID와 버전 두 개입니다."
assert set(answer_table.columns) == set(metric_names), "answer_table의 열은 지표 이름 세 개입니다."

for row in score_rows:
    table_value = metric_tables[row["지표"]].loc[row["질문 ID"], row["버전"]]
    answer_value = answer_table.loc[(row["질문 ID"], row["버전"]), row["지표"]]
    if pd.isna(row["점수"]):
        assert pd.isna(table_value) and pd.isna(answer_value), "NaN 점수는 두 표 모두 NaN으로 둡니다."
    else:
        assert abs(table_value - round(row["점수"], 2)) < 1e-9, f"{row['지표']}: 그 지표 행만 골라 점수를 소수 둘째 자리로 반올림하세요."
        assert answer_value == row["점수"], "answer_table은 반올림하지 않은 점수를 그대로 담습니다."
print("표 확인 완료: 지표별 표", len(metric_tables), "개, 답변별 점수표", answer_table.shape)

In [ ]:
# 지표별 표 세 개를 옆으로 붙여 한 번에 봅니다 (열: 지표 아래 버전)
display(pd.concat(metric_tables, axis=1))

같은 질문 행에서 두 버전의 점수가 크게 갈린 칸을 찾아 두세요. 어느 버전의 어떤 답변이 원인인지는 4절의 라벨과 원문으로 확인합니다.

## 4. 세 지표를 함께 읽어 답변마다 라벨을 붙이세요

지표 하나로는 결함 종류를 가릴 수 없으므로(교안 01 7절), 교안 01 6-3의 두 지표 함께 읽기 표를 규칙으로 옮겨 결함 종류를 모르는 답변에 라벨을 붙입니다. 라벨 이름은 6-3 표의 칸 이름(교안 01 본실습의 변형 이름)과 같습니다.

**기준값과 읽는 법**
- Faithfulness가 0.8 미만이면 검색 문서를 벗어난 답변, Answer Correctness가 0.5 미만이면 정답 답변과 다른 답변으로 봅니다.
- Faithfulness 0.8은 주장 4개 가운데 하나만 근거가 없어도 0.75가 되어 걸리는 값입니다.
- Answer Correctness 0.5는 정답 답변의 주장을 하나도 맞히지 못하면 사실 점수가 0이라 0.25 x 의미 유사도만 남는다는 점(교안 01 6-1)에서 정했습니다. 의미 유사도가 1이어도 0.25라 이 기준 아래로 떨어집니다.
- Answer Relevancy는 엇나간 답변도 0 가까이 떨어지지 않으므로(교안 01 5-2) 절댓값 기준을 둘 수 없어 라벨 규칙에 쓰지 않습니다. 버전 평균을 비교할 때 참고합니다.

**규칙** (위에서부터 검사해 처음 맞는 규칙의 라벨)

| 순서 | 조건 | 라벨 | 교안 01 6-3 표의 칸 |
|---|---|---|---|
| 1 | Faithfulness가 NaN | `원문 확인` | 4-2: 답변에서 주장을 하나도 뽑지 못함 |
| 2 | Faithfulness < 0.8 이고 Answer Correctness >= 0.5 | `근거에_없는_내용_추가` | 문서를 벗어났지만 정답 답변의 핵심은 담음 |
| 3 | Faithfulness < 0.8 이고 Answer Correctness < 0.5 | `숫자_조건_바꿈` | 문서를 벗어나고 정답 답변과도 다름 |
| 4 | Answer Correctness < 0.5 (Faithfulness >= 0.8) | `질문과_다른_이야기` | 받은 문서대로 답했지만 질문이 묻는 내용이 아님 |
| 5 | 나머지 | `근거대로` | 문서에 충실하고 정답 답변과도 일치 |

**작성:** 함수 `read_label(scores)`

**목적:** 교안 01 9-3의 판정 셀처럼 표를 읽는 규칙을 코드로 옮겨, 답변마다 같은 기준으로 라벨을 붙입니다.

**구현할 처리**
1. `scores`는 답변 하나의 세 점수(`answer_table`의 한 행)이고 `scores["Faithfulness"]`처럼 지표 이름으로 읽습니다(골격에 있음).
2. 위 표의 규칙을 순서대로 검사해 처음 맞는 라벨 문자열을 `return` 합니다. 기준값은 셀 위쪽 상수 `FAITHFULNESS_MIN`, `CORRECTNESS_MIN`을 씁니다.
3. "미만"은 `<`입니다. 기준값과 같은 점수는 낮다고 보지 않습니다.

**반환값:** 라벨 문자열 하나

In [ ]:
# [작성] 세 점수로 판독 라벨을 반환하는 함수를 완성하세요.
FAITHFULNESS_MIN = 0.8  # 이보다 낮으면 검색 문서를 벗어난 답변
CORRECTNESS_MIN = 0.5   # 이보다 낮으면 정답 답변과 다른 답변


def read_label(scores):
    """답변 하나의 세 점수를 교안 01 6-3 표의 규칙 순서대로 읽어 라벨을 반환합니다."""
    faithfulness_score = scores["Faithfulness"]
    correctness_score = scores["Answer Correctness"]

    # 1. Faithfulness가 NaN
    ...

    # 2~3. Faithfulness가 기준보다 낮음 (Answer Correctness로 둘을 나눔)
    ...

    # 4. 검색 문서에는 충실한데 Answer Correctness가 기준보다 낮음
    ...

    # 5. 나머지
    return "근거대로"

In [ ]:
# [자가채점] 가짜 고정 점수로 규칙의 순서와 경계를 확인합니다. 실제 점수는 쓰지 않습니다.
def fake_scores(faithfulness_score, relevancy_score, correctness_score):
    """자가채점용 가짜 점수 한 행을 만듭니다(answer_table의 한 행과 같은 열)."""
    return pd.Series({"Faithfulness": faithfulness_score, "Answer Relevancy": relevancy_score, "Answer Correctness": correctness_score})


# (가짜 점수, 기대 라벨, 틀렸을 때 볼 규칙)
fake_cases = [
    (fake_scores(float("nan"), 0.6, 0.1), "원문 확인", "Faithfulness가 NaN이면 다른 점수와 상관없이 먼저 원문 확인입니다(규칙 1). NaN은 pd.isna로 확인하세요."),
    (fake_scores(0.5, 0.6, 0.7), "근거에_없는_내용_추가", "Faithfulness가 낮고 Answer Correctness가 기준 이상이면 근거에_없는_내용_추가입니다(규칙 2)."),
    (fake_scores(0.5, 0.6, 0.5), "근거에_없는_내용_추가", "Answer Correctness가 기준값(0.5)과 같으면 낮다고 보지 않습니다."),
    (fake_scores(0.3, 0.8, 0.2), "숫자_조건_바꿈", "Faithfulness와 Answer Correctness가 둘 다 낮으면 숫자_조건_바꿈입니다(규칙 3)."),
    (fake_scores(0.8, 0.6, 0.9), "근거대로", "Faithfulness가 기준값(0.8)과 같으면 낮다고 보지 않습니다."),
    (fake_scores(1.0, 0.5, 0.3), "질문과_다른_이야기", "Faithfulness는 높은데 Answer Correctness가 낮으면 질문과_다른_이야기입니다(규칙 4)."),
    (fake_scores(0.79, 0.9, 0.3), "숫자_조건_바꿈", "Faithfulness를 먼저 봅니다. 0.79는 기준보다 낮으므로 규칙 4가 아니라 규칙 3입니다."),
    (fake_scores(1.0, 0.9, 0.95), "근거대로", "두 점수가 모두 기준 이상이면 근거대로입니다(규칙 5)."),
    (fake_scores(1.0, 0.1, 0.9), "근거대로", "Answer Relevancy는 라벨 규칙에 쓰지 않습니다. 낮아도 라벨이 바뀌지 않습니다."),
]
for scores, expected_label, rule in fake_cases:
    label = read_label(scores)
    assert isinstance(label, str), "라벨 문자열을 return 하세요. `...`를 채웠는지 확인하세요."
    assert label == expected_label, f"점수 {dict(scores)}: {label!r} 대신 {expected_label!r}이어야 합니다. {rule}"
print("라벨 함수 확인 완료: 가짜 점수", len(fake_cases), "개 통과")

아래 셀은 `answer_table`의 답변마다 `read_label`을 적용해 라벨 표 `label_rows`를 만듭니다. 5절은 이 표로 버전마다 라벨을 셉니다.

In [ ]:
# 답변마다 라벨을 붙여 한 표로 모읍니다
label_rows = []
for (question_id, version), scores in answer_table.iterrows():
    label_rows.append({
        "질문 ID": question_id,
        "버전": version,
        "Faithfulness": round(scores["Faithfulness"], 2),
        "Answer Relevancy": round(scores["Answer Relevancy"], 2),
        "Answer Correctness": round(scores["Answer Correctness"], 2),
        "라벨": read_label(scores),
    })
display(pd.DataFrame(label_rows))

라벨은 점수로 좁힌 추정입니다. 라벨이 `근거대로`가 아닌 답변은 원문을 정답 답변과 나란히 놓고, 어느 문장이 문제인지 확인합니다. 확인한 뒤 고칠 곳은 라벨마다 다릅니다(교안 01 6-3).

- **`근거에_없는_내용_추가`, `숫자_조건_바꿈`:** 답변이 받은 문서를 벗어났으므로 답변 생성 단계(프롬프트의 지시, 모델)를 고칩니다.
- **`질문과_다른_이야기`:** 답변은 받은 문서대로 말했지만 질문이 묻는 부분이 아닙니다. 프롬프트가 질문의 어느 부분을 놓치게 했는지 보고, 받은 문서가 질문과 맞지 않거나 낡지 않았는지도 확인합니다(교안 01 9-3의 `문서 확인`).
- **`원문 확인`:** 점수를 계산하지 못한 답변이라 원문부터 읽습니다(교안 01 4-2).

In [ ]:
# (질문 ID, 버전)으로 샘플을 찾는 dict
sample_by_key = {}
for sample in samples:
    sample_by_key[(sample["질문 ID"], sample["버전"])] = sample

# 라벨이 근거대로가 아닌 답변의 원문을 정답 답변과 나란히 봅니다
for row in label_rows:
    if row["라벨"] == "근거대로":
        continue
    sample = sample_by_key[(row["질문 ID"], row["버전"])]
    print(row["질문 ID"], row["버전"], "| 라벨:", row["라벨"], "| F·AR·AC:", row["Faithfulness"], row["Answer Relevancy"], row["Answer Correctness"])
    print("질문:", sample["user_input"])
    print("답변:", sample["response"])
    print("정답 답변:", sample["reference"])
    print()

## 5. 버전별 평균과 문제 답변 수를 비교하세요

출시 여부는 버전 전체를 보고 정하므로, 지표 평균과 라벨 개수를 버전마다 한 행에 모읍니다. 평균은 점수가 크게 떨어진 답변 하나에 끌려가고 라벨 개수는 그 크기를 보지 않으므로 둘을 함께 둡니다.

**작성:** 버전 비교표 `version_summary`

**목적:** 교안 01 7절의 평균 표(`pivot_table`)를 버전별로 만들고, 4절의 라벨을 버전마다 셉니다.

**구현할 처리**
1. `score_df`로 행은 `버전`, 열은 `지표`, 값은 `점수`의 평균인 표를 `pivot_table`로 만들고 소수 둘째 자리로 반올림합니다. NaN 점수는 평균에서 빠집니다.
2. `label_rows`를 읽어 버전마다 라벨이 `근거대로`가 아닌 답변 수를 `problem_counts`에 셉니다.
3. 같은 반복에서 라벨이 `근거에_없는_내용_추가` 또는 `숫자_조건_바꿈`인 답변 수를 `grounding_counts`에 셉니다. 두 라벨은 검색 문서를 벗어난 답변이라, 사용자가 안내문에 없는 복용 정보를 믿게 됩니다.
4. 두 dict를 `문제 답변 수`, `근거 이탈 수` 열로 붙입니다(골격에 있음).

**반환값:** `version_summary`(행 `v1`·`v2`, 열은 지표 세 개와 `문제 답변 수`, `근거 이탈 수`)

In [ ]:
# [작성] 버전별 지표 평균에 문제 답변 수와 근거 이탈 수를 붙인 표를 만드세요.

# 1. 행은 버전, 열은 지표인 평균 표 (소수 둘째 자리)
version_summary = ...

# 2~3. 버전마다 라벨이 근거대로가 아닌 답변 수와 근거 이탈 답변 수 세기
problem_counts = {"v1": 0, "v2": 0}
grounding_counts = {"v1": 0, "v2": 0}
grounding_labels = ["근거에_없는_내용_추가", "숫자_조건_바꿈"]
for row in label_rows:
    ...

# 4. 버전 이름이 같은 행에 개수를 붙입니다
version_summary["문제 답변 수"] = pd.Series(problem_counts)
version_summary["근거 이탈 수"] = pd.Series(grounding_counts)

In [ ]:
# [자가채점] 평균은 score_rows에서 하나씩 더해, 개수는 라벨 표를 걸러 세어 대조합니다.
assert isinstance(version_summary, pd.DataFrame) and list(version_summary.index) == ["v1", "v2"], "version_summary의 행은 v1, v2입니다. `...`를 채웠는지 확인하세요."
for metric_name in metric_names:
    assert metric_name in version_summary.columns, f"version_summary에 {metric_name} 평균 열이 있어야 합니다(열은 지표)."
    for version in ["v1", "v2"]:
        total = 0
        count = 0
        for row in score_rows:
            if row["지표"] == metric_name and row["버전"] == version and not pd.isna(row["점수"]):
                total += row["점수"]
                count += 1
        assert abs(version_summary.loc[version, metric_name] - round(total / count, 2)) < 1e-9, f"{version} {metric_name}: 그 버전·지표 점수의 평균(NaN 제외)을 소수 둘째 자리로 반올림하세요."

label_check = pd.DataFrame(label_rows)
is_problem = label_check["라벨"] != "근거대로"
is_grounding = label_check["라벨"].isin(["근거에_없는_내용_추가", "숫자_조건_바꿈"])
for version in ["v1", "v2"]:
    in_version = label_check["버전"] == version
    assert version_summary.loc[version, "문제 답변 수"] == int((in_version & is_problem).sum()), f"{version}: 문제 답변 수는 라벨이 근거대로가 아닌 답변 수입니다."
    assert version_summary.loc[version, "근거 이탈 수"] == int((in_version & is_grounding).sum()), f"{version}: 근거 이탈 수는 근거에_없는_내용_추가와 숫자_조건_바꿈 라벨의 수입니다."
print("버전 비교표 확인 완료")

In [ ]:
# 버전별 지표 평균과 라벨 개수
display(version_summary)

## 6. 출시 판정 규칙을 함수로 옮기세요

표를 보고 그때그때 판정하면 다음에 프롬프트를 바꿀 때 기준이 달라지므로, 상담 도우미 팀이 정한 규칙을 함수로 고정해 같은 골든셋으로 다시 평가할 때마다 같은 방식으로 판정합니다.

**출시 판정 규칙**
- `v2`의 문제 답변 수가 기준선 `v1`보다 **적어야** 합니다. 같으면 바꿀 이유가 없습니다.
- `v2`의 근거 이탈 수가 기준선 `v1`보다 **많으면 안 됩니다.** 안내문에 없는 복용 정보를 말하는 답변이 늘면 안 됩니다.
- 두 조건을 모두 만족하면 `출시`, 하나라도 어기면 `보류`입니다. 지표 평균은 판정에 쓰지 않고 결정 기록의 근거로만 씁니다.
- `원문 확인` 라벨(Faithfulness가 NaN)도 `근거대로`가 아니므로 문제 답변으로 셉니다.

**작성:** 함수 `decide_release(summary)`

**목적:** 교안 01 8-1의 기준선 비교를 라벨 개수로 옮겨, 개선안 `v2`가 기준선 `v1`보다 문제 답변은 줄고 근거 이탈은 늘지 않았는지를 판정합니다.

**구현할 처리**
1. `summary`는 5절의 `version_summary`와 같은 모양의 표입니다. `summary.loc["v2", "문제 답변 수"]`처럼 읽습니다.
2. 첫째 조건(문제 답변 수가 더 적은가)과 둘째 조건(근거 이탈 수가 더 많지 않은가)을 각각 `True`/`False`로 구합니다.
3. 둘 다 `True`이면 `"출시"`, 아니면 `"보류"`를 반환합니다.

**반환값:** `"출시"` 또는 `"보류"`

In [ ]:
# [작성] 버전 비교표로 v2 출시 여부를 반환하는 함수를 완성하세요.
def decide_release(summary):
    """버전 비교표의 문제 답변 수와 근거 이탈 수로 v2 출시 여부("출시" 또는 "보류")를 반환합니다."""
    # 1. v2의 문제 답변 수가 v1보다 적은가
    fewer_problems = ...

    # 2. v2의 근거 이탈 수가 v1보다 많지 않은가
    no_more_grounding = ...

    # 3. 두 조건을 모두 만족하면 출시, 아니면 보류
    ...

In [ ]:
# [자가채점] 가짜 비교표로 규칙의 경계를 확인합니다. 실제 비교표는 쓰지 않습니다.
def fake_summary(v1_problems, v1_grounding, v2_problems, v2_grounding, v1_faithfulness, v2_faithfulness):
    """자가채점용 가짜 버전 비교표를 만듭니다(version_summary와 같은 행·열 이름)."""
    return pd.DataFrame({
        "Faithfulness": {"v1": v1_faithfulness, "v2": v2_faithfulness},
        "문제 답변 수": {"v1": v1_problems, "v2": v2_problems},
        "근거 이탈 수": {"v1": v1_grounding, "v2": v2_grounding},
    })


# (표 이름, 가짜 비교표, 기대 판정, 틀렸을 때 볼 규칙)
fake_tables = [
    ("가 문제 답변이 줄고 근거 이탈은 그대로", fake_summary(3, 1, 2, 1, 0.9, 0.8), "출시", "문제 답변이 줄고 근거 이탈이 늘지 않으면 출시입니다. 근거 이탈이 같은 것은 허용합니다."),
    ("나 문제 답변 수가 같음", fake_summary(3, 1, 3, 0, 0.7, 0.9), "보류", "문제 답변 수가 같으면 첫째 조건(더 적어야 함)을 어겨 보류입니다."),
    ("다 문제 답변은 줄었지만 근거 이탈이 늘어남", fake_summary(3, 1, 2, 2, 0.7, 0.9), "보류", "근거 이탈 수가 늘면 문제 답변이 줄어도 보류입니다."),
    ("라 평균은 v2가 높지만 문제 답변이 늘어남", fake_summary(2, 1, 4, 1, 0.6, 0.9), "보류", "지표 평균은 판정에 쓰지 않습니다. 문제 답변 수로 판정하세요."),
    ("마 평균은 v2가 낮지만 규칙을 만족", fake_summary(4, 2, 1, 1, 0.9, 0.7), "출시", "지표 평균은 판정에 쓰지 않습니다. 두 조건을 모두 만족하면 출시입니다."),
]
for table_name, summary, expected_decision, rule in fake_tables:
    decision = decide_release(summary)
    assert decision in ("출시", "보류"), "\"출시\" 또는 \"보류\"를 return 하세요. `...`를 채웠는지 확인하세요."
    assert decision == expected_decision, f"[{table_name}] {decision!r} 대신 {expected_decision!r}이어야 합니다. {rule}"
print("판정 함수 확인 완료: 가짜 표", len(fake_tables), "개 통과")

In [ ]:
# 실제 버전 비교표로 판정합니다
decision = decide_release(version_summary)
print("v2 판정:", decision, "| 문제 답변 수 v1·v2:", int(version_summary.loc["v1", "문제 답변 수"]), int(version_summary.loc["v2", "문제 답변 수"]), "| 근거 이탈 수 v1·v2:", int(version_summary.loc["v1", "근거 이탈 수"]), int(version_summary.loc["v2", "근거 이탈 수"]))

## 7. 출시 결정 기록을 쓰세요

**작성:** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 제목과 네 칸(상태·맥락·결정·결과)을 채웁니다. 제목은 결정이 드러나게 씁니다(예: "답변 프롬프트 v2 출시"). 맥락은 정해야 할 문제와 평가 방법을 2~3문장으로 쓰고 결정은 쓰지 않습니다. 결정은 "…한다."로 시작하고, 아래 `근거:`에 줄마다 숫자 하나 이상을 씁니다. 결과 아래에는 `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(바뀌는 것 -> 다시 할 일)을 씁니다. 숫자는 자신이 실행한 표와 출력에서 옮기세요. 점수는 실행마다 조금씩 달라져 다른 실행과 라벨이 다를 수 있으니 자기 결과로 씁니다.

**목적:** 평균과 라벨 개수가 각각 무엇을 말하는지 숫자로 설명하고, 판정 뒤에 고칠 곳을 남깁니다.

**설명할 내용**
1. 상태: 제안·채택·거절·대체됨 가운데 하나(이 결정을 받아들였으면 `채택`)
2. 맥락: 정해야 할 것(`v2` 출시), 평가 방법(골든셋 질문 수, 세 지표, 라벨 기준값과 판정 규칙)
3. 결정: 판정을 "…한다."로 쓰고, `근거:`에 아래를 씁니다.
    - 버전별 지표 평균 세 개와 문제 답변 수·근거 이탈 수
    - 평균과 라벨 개수가 같은 결론을 주는지. 다르다면 어느 답변의 점수가 평균을 움직였는지(평균은 점수가 떨어진 크기를, 라벨 개수는 결함이 있는 답변 수를 봅니다)
    - `v2`가 고친 답변과 새로 생긴 문제 답변(질문 ID, 라벨, 답변 원문에서 문제가 된 문장)
    - 새로 생긴 문제 답변을 사용자가 그대로 따르면 무슨 일이 생기는지
4. 결과: `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`을 둘 이상(프롬프트 수정, 골든셋 보강, 다시 실행 등). 판정이 `보류`면 그중 하나는 문제 답변을 고칠 방법으로 씁니다. 4절 라벨에 따라 프롬프트의 어느 지시를 어떻게 바꿀지, 또는 검색 문서를 바꿀지 적습니다
5. 라벨이 흔들릴 수 있는 답변: 기준값(Faithfulness 0.8, Answer Correctness 0.5)과 0.1 안쪽인 답변을 `근거:`에 적고(없으면 없다고 쓰고 기준값에 가장 가까운 답변을 적습니다), 출시 판단 전에는 2절을 두세 번 실행해 라벨이 바뀌는지 확인한다는 계획을 `다시 검토할 조건:`에 씁니다.

#### 결정 기록: (결정이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제(v2 출시), 평가 방법(골든셋 질문 수, 세 지표, 라벨 기준값과 판정 규칙). 결정은 쓰지 않음)
- **결정**: (판정을 한 문장으로: …한다)
    - 근거: (줄마다 숫자 하나 이상: 버전별 평균과 라벨 개수, 평균과 개수가 다른 이유, v2가 고친 답변과 새로 생긴 문제 답변, 새 문제 답변의 위험, 기준값과 0.1 안쪽인 답변)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 것 -> 다시 할 일)